# D006 — 모두마켓 월간 웹 로그: 도구 선택

> 위쪽의 **모두마켓 데이터 생성 셀**을 먼저 실행해서 `logs_csv`가 준비된 상태에서 아래를 순서대로 실행하세요.

In [ ]:
import os, gc, time, threading
import numpy as np
import pandas as pd
import polars as pl
import psutil
import matplotlib.pyplot as plt

def rss_mb():
    return psutil.Process(os.getpid()).memory_info().rss / 1024 / 1024

def measure(func, *args, warmup=True, **kwargs):
    """func 실행 → (결과, 소요 시간 초, 메모리 피크 증가량 MB)
    - warmup: 첫 실행의 일회성 비용(라이브러리 초기화 등)을 빼기 위해 한 번 미리 실행
    - 피크: 실행 중 0.005초마다 RSS를 재서 가장 높았던 값을 기록"""
    if warmup:
        func(*args, **kwargs)
    gc.collect()
    before = rss_mb()
    peak = [before]
    running = [True]

    def sampler():
        while running[0]:
            peak[0] = max(peak[0], rss_mb())
            time.sleep(0.005)

    th = threading.Thread(target=sampler)
    th.start()
    t0 = time.perf_counter()
    result = func(*args, **kwargs)
    elapsed = time.perf_counter() - t0
    running[0] = False
    th.join()
    return result, elapsed, max(peak[0] - before, 0)

USECOLS = ["page", "device", "status_code", "response_ms"]

## 시나리오 1 — pandas + dtype 최적화
필요한 4개 칼럼만 읽고(`usecols`), 글자는 `category`, 숫자는 작은 정수형으로 읽어 메모리를 최소화합니다.

In [ ]:
def analyze_pandas():
    df = pd.read_csv(
        logs_csv,
        usecols=USECOLS,
        dtype={"page": "category", "device": "category",
               "status_code": "int16", "response_ms": "int32"},
    )
    page_avg = df.groupby("page", observed=True)["response_ms"].mean().round(2)
    device_share = (df["device"].value_counts(normalize=True) * 100).round(1)
    error_by_page = df[df["status_code"] >= 400].groupby("page", observed=True).size()
    return {
        "page_avg": page_avg.sort_index(),
        "device_share": device_share.sort_index(),
        "error_by_page": error_by_page.sort_index(),
    }

res_pd, t_pd, m_pd = measure(analyze_pandas)
print(f"pandas + dtype : {t_pd:.2f}초, 메모리 +{m_pd:.1f}MB")

## 시나리오 2 — 청크 처리
청크마다 **합계·개수**만 모으고, 마지막에 전체 합 ÷ 전체 개수로 평균을 냅니다. (평균의 평균은 청크마다 개수가 달라 틀림)

In [ ]:
def analyze_chunked(chunksize=50_000):
    page_parts, device_parts, error_parts = [], [], []
    for chunk in pd.read_csv(
        logs_csv, chunksize=chunksize, usecols=USECOLS,
        dtype={"page": "category", "device": "category",
               "status_code": "int16", "response_ms": "int32"},
    ):
        # 청크마다 '합칠 수 있는 중간값'만 모음: 합계·개수
        page_parts.append(chunk.groupby("page", observed=True)["response_ms"].agg(["sum", "count"]))
        device_parts.append(chunk["device"].value_counts())
        error_parts.append(chunk[chunk["status_code"] >= 400].groupby("page", observed=True).size())

    # reduce: 합계끼리, 개수끼리 더한 뒤 마지막에 나누기 (평균의 평균 X)
    page_tot = pd.concat(page_parts).groupby(level=0).sum()
    page_avg = (page_tot["sum"] / page_tot["count"]).round(2)

    device_cnt = pd.concat(device_parts).groupby(level=0).sum()
    device_share = (device_cnt / device_cnt.sum() * 100).round(1)

    error_by_page = pd.concat(error_parts).groupby(level=0).sum()

    return {
        "page_avg": page_avg.sort_index(),
        "device_share": device_share.sort_index(),
        "error_by_page": error_by_page.sort_index(),
    }

res_ck, t_ck, m_ck = measure(analyze_chunked, 50_000)
print(f"pandas chunked : {t_ck:.2f}초, 메모리 +{m_ck:.1f}MB")

## 시나리오 3 — Polars lazy
`scan_csv`(읽기) → `filter`(필터) → `group_by`/`agg`(집계) → `collect`(실행). 세 쿼리를 `collect_all`로 한 번에 실행합니다.

In [ ]:
def analyze_polars():
    lf = pl.scan_csv(logs_csv).select(USECOLS)      # 읽기 (아직 실행 X)

    q_page = (lf.group_by("page")
                .agg(pl.col("response_ms").mean().round(2).alias("avg_ms")))
    q_device = (lf.group_by("device")
                  .agg(pl.len().alias("n"))
                  .with_columns((pl.col("n") / pl.col("n").sum() * 100).round(1).alias("share")))
    q_error = (lf.filter(pl.col("status_code") >= 400)          # 필터
                 .group_by("page")                              # 집계
                 .agg(pl.len().alias("errors")))

    page, device, error = pl.collect_all([q_page, q_device, q_error])   # collect (한 번에 실행)

    to_s = lambda df, k, v: pd.Series(df[v].to_list(), index=df[k].to_list()).sort_index()
    return {
        "page_avg": to_s(page, "page", "avg_ms"),
        "device_share": to_s(device, "device", "share"),
        "error_by_page": to_s(error, "page", "errors"),
    }

res_pl, t_pl, m_pl = measure(analyze_polars)
print(f"Polars lazy    : {t_pl:.2f}초, 메모리 +{m_pl:.1f}MB")

## 결과 비교 — 세 가지 방식 한 자리에

In [ ]:
def same(a, b):
    return all(np.allclose(a[k].astype(float).values, b[k].astype(float).reindex(a[k].index).values)
               for k in a)

print("세 방식 결과 동일:", same(res_pd, res_ck) and same(res_pd, res_pl))

summary = pd.DataFrame({
    "방식": ["pandas + dtype", "pandas chunked", "Polars lazy"],
    "소요 시간(초)": [round(t_pd, 2), round(t_ck, 2), round(t_pl, 2)],
    "메모리 증가(MB)": [round(m_pd, 1), round(m_ck, 1), round(m_pl, 1)],
})
print(summary.to_string(index=False))

print("\n[페이지별 평균 응답 시간]\n", res_pd["page_avg"].sort_values(ascending=False))
print("\n[디바이스 점유율 %]\n", res_pd["device_share"])
print("\n[에러 건수 by 페이지]\n", res_pd["error_by_page"].sort_values(ascending=False))

In [ ]:
# [도식] 세 가지 방식의 시간·메모리 비교
methods = ["pandas+dtype", "Chunked", "Polars lazy"]
times = [t_pd, t_ck, t_pl]
peaks = [m_pd, m_ck, m_pl]
colors = ["#4C72B0", "#55A868", "#C44E52"]

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].bar(methods, times, color=colors)
axes[0].set_title("Time (seconds, lower is better)")
axes[0].set_ylabel("seconds")
for i, v in enumerate(times):
    axes[0].text(i, v, f"{v:.2f}s", ha="center", va="bottom")

axes[1].bar(methods, peaks, color=colors)
axes[1].set_title("Peak memory increase (MB, lower is better)")
axes[1].set_ylabel("MB")
for i, v in enumerate(peaks):
    axes[1].text(i, v, f"{v:.1f}MB", ha="center", va="bottom")

plt.tight_layout()
plt.show()

## 📊 도구 선택 보고서
아래 셀은 **이 노트북에서 측정한 숫자로 빈칸을 자동으로 채워** 보고서를 출력합니다. 4번 정당화 문단은 출력된 숫자를 보고 필요하면 문장을 다듬으세요.

In [ ]:
file_mb = logs_csv.stat().st_size / 1024 / 1024
n_rows = pl.scan_csv(logs_csv).select(pl.len()).collect().item()

page_avg = res_pd["page_avg"]
share = res_pd["device_share"]
errors = res_pd["error_by_page"]
top_page, top_ms = page_avg.idxmax(), page_avg.max()
err_page, err_cnt = errors.idxmax(), errors.max()

fastest = summary.loc[summary["소요 시간(초)"].idxmin(), "방식"]
lightest = summary.loc[summary["메모리 증가(MB)"].idxmin(), "방식"]

report = f"""# 모두마켓 월간 로그 — 도구 선택 보고서

## 1. 데이터 개요
- 파일: web_logs.csv
- 크기: {file_mb:.1f} MB, {n_rows:,} 행
- 주요 컬럼: ts / user_id / page / device / status_code / response_ms / bytes_sent

## 2. 측정 결과
| 방식 | 소요 시간 | 메모리 피크 |
|---|---|---|
| pandas + dtype | {t_pd:.2f}초 | {m_pd:.1f}MB |
| pandas chunked | {t_ck:.2f}초 | {m_ck:.1f}MB |
| Polars lazy    | {t_pl:.2f}초 | {m_pl:.1f}MB |

## 3. 분석 결과 요약
- 페이지별 평균 응답 시간 최고: {top_page} ({top_ms:.2f}ms)
- 디바이스 점유율: mobile {share['mobile']:.1f}%, desktop {share['desktop']:.1f}%, tablet {share['tablet']:.1f}%
- 에러 1위 페이지: {err_page} ({err_cnt:,}건)

## 4. 도구 선택 정당화 (한 단락)
이 분석에는 **pandas + dtype 최적화**를 선택했습니다. 이유는
(1) 측정 결과 가장 빠른 방식은 {fastest}, 메모리가 가장 적은 방식은 {lightest}였지만, 세 방식 모두 1초 안팎에 끝나고 메모리 차이도 수십 MB 이내라 {n_rows:,}행·{file_mb:.0f}MB 규모에서는 체감 차이가 거의 없으며,
(2) 팀 환경상 pandas가 가장 익숙하고 시각화(matplotlib)·머신러닝(sklearn)과 바로 연결되어 코드 리뷰와 유지보수가 쉽고,
(3) 청크 처리는 합계·개수를 손으로 합치는 로직이 필요해 코드가 길어지고 중앙값 같은 분석으로 확장하기 어렵기 때문입니다.

## 5. 다음 단계 제안
- 데이터 크기가 1천만 행(수 GB)까지 늘어나면 Polars lazy(scan_csv/scan_parquet)로 전환을 고려.
- 시각화·검증은 D+008에서 matplotlib/seaborn을 활용해 진행.
"""
print(report)

## 시나리오 4 — 도전 과제(선택): chunksize 비교
청크가 작을수록 메모리는 적게 쓰지만 반복 횟수가 늘어 느려지고, 클수록 빠르지만 메모리를 더 씁니다.

In [ ]:
import tracemalloc

# RSS는 한 번 늘어난 메모리를 바로 돌려주지 않아 연속 측정에 둔감하므로,
# 여기서는 파이썬/넘파이가 실제로 잡은 메모리의 피크를 재는 tracemalloc을 씁니다.
rows = []
for cs in [10_000, 50_000, 100_000]:
    gc.collect()
    tracemalloc.start()
    t0 = time.perf_counter()
    analyze_chunked(cs)
    elapsed = time.perf_counter() - t0
    _, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    rows.append({"chunksize": cs, "시간(초)": round(elapsed, 2), "메모리 피크(MB)": round(peak / 1024**2, 1)})
print(pd.DataFrame(rows).to_string(index=False))

> **측정 참고:** 메모리는 실행 중 RSS를 0.005초마다 재서 가장 높았던 값(피크)에서 시작 전 값을 뺀 근사치입니다. 첫 실행의 라이브러리 초기화 비용을 빼려고 한 번 워밍업 후 측정했고, 같은 프로세스에서 연달아 재기 때문에 실행 순서나 환경에 따라 숫자가 조금씩 달라질 수 있습니다.